## Overview

`Conv_Decomposer` factorizes Conv2d layers into smaller convolutions using 4 different mathematical decompositions. Each trades off compression, accuracy, and inference overhead differently.

In [ ]:
#| include: false
from fastai.vision.all import *
from fasterai.misc.all import *
import torch
import torch.nn as nn

## 1. Setup and Data

In [ ]:
path = untar_data(URLs.PETS)
files = get_image_files(path/"images")
def label_func(f): return f[0].isupper()
dls = ImageDataLoaders.from_name_func(path, files, label_func, item_tfms=Resize(64))

## 2. Train the Model

We use ResNet-18 — a Conv-heavy model where Tucker decomposition is effective:

In [ ]:
learn = vision_learner(dls, resnet18, metrics=accuracy)
learn.unfreeze()
learn.fit(3)

## 3. Compare Decomposition Methods

Let's decompose the same model with all 4 methods and compare:

In [ ]:
import copy

def count_params(model):
    return sum(p.numel() for p in model.parameters())

original_params = count_params(learn.model)
decomposer = Conv_Decomposer()

print(f"{'Method':<10} {'Layers':<8} {'Params':>10} {'Compression':>12} {'Structure'}")
print("-" * 70)
print(f"{'original':<10} {'—':<8} {original_params:>10,} {'1.0x':>12} {'Conv2d(C_in, C_out, K×K)'}")

for method in ['svd', 'tucker', 'spatial', 'cp']:
    model_dec = decomposer.decompose(copy.deepcopy(learn.model), 0.5, method=method)
    params = count_params(model_dec)
    ratio = original_params / params
    
    structures = {
        'svd':     'K×K + 1×1',
        'tucker':  '1×1 + K×K + 1×1',
        'spatial': 'K×1 + 1×K',
        'cp':      '1×1 + K×1(dw) + 1×K(dw) + 1×1',
    }
    n_layers = {'svd': 2, 'tucker': 3, 'spatial': 2, 'cp': 4}
    
    print(f"{method:<10} {n_layers[method]:<8} {params:>10,} {ratio:>11.1f}x {structures[method]}")

### How Each Method Decomposes a Conv2d(64, 128, 3×3)

**SVD** (2 layers) — decomposes output channels:
```
Conv2d(64, R, 3×3)  →  Conv2d(R, 128, 1×1)
```

**Tucker** (3 layers) — decomposes both channel dimensions:
```
Conv2d(64, R_in, 1×1)  →  Conv2d(R_in, R_out, 3×3)  →  Conv2d(R_out, 128, 1×1)
```

**Spatial** (2 layers) — decomposes the kernel spatially:
```
Conv2d(64, 128×R, 3×1)  →  Conv2d(128×R, 128, 1×3, groups=128)
```

**CP** (4 layers) — decomposes channels AND spatial:
```
Conv2d(64, R, 1×1)  →  Conv2d(R, R, 3×1, dw)  →  Conv2d(R, R, 1×3, dw)  →  Conv2d(R, 128, 1×1)
```

Each targets a different source of redundancy. Tucker is the best general-purpose choice; CP gives maximum compression but may need more fine-tuning.

## 4. Accuracy Impact (Before Fine-Tuning)

Each method has a different reconstruction error — let's measure accuracy drop:

In [ ]:
baseline = Learner(dls, learn.model, metrics=accuracy).validate()[1]
print(f"{'Method':<10} {'Accuracy':>10} {'vs Baseline':>12}")
print("-" * 35)
print(f"{'original':<10} {baseline*100:>9.1f}% {'':>12}")

for method in ['svd', 'tucker', 'spatial', 'cp']:
    model_dec = decomposer.decompose(copy.deepcopy(learn.model), 0.5, method=method)
    acc = Learner(dls, model_dec, metrics=accuracy).validate()[1]
    print(f"{method:<10} {acc*100:>9.1f}% {(acc-baseline)*100:>+11.1f}%")

The accuracy drops after decomposition — Tucker decomposition is an **approximation**, not exact. Fine-tuning recovers most of the accuracy:

```python
new_learn.fit_one_cycle(3, 1e-4)  # Fine-tune with small learning rate
```

## 5. Controlling Compression

The `percent_removed` parameter controls how much rank is removed per mode:

| percent_removed | Rank Kept | Compression | Accuracy Impact |
|-----------------|-----------|-------------|-----------------|
| `0.0` | 100% | ~1x (near-exact) | Minimal |
| `0.3` | 70% | ~1.5-2x | Low |
| `0.5` | 50% | ~2-4x | Moderate |
| `0.7` | 30% | ~4-8x | Significant |

```python
# Light compression — minimal accuracy loss
light = Conv_Decomposer().decompose(model, percent_removed=0.3)

# Heavy compression — needs fine-tuning
heavy = Conv_Decomposer().decompose(model, percent_removed=0.7)
```

## 6. Combining with Other Techniques

Tucker decomposition works well as a first step before other compressions:

```python
from fasterai.misc.all import Conv_Decomposer, BN_Folder

# 1. Fold BatchNorm into Conv layers
model = BN_Folder().fold(model)

# 2. Decompose Conv layers
model = Conv_Decomposer().decompose(model, percent_removed=0.5)

# 3. Fine-tune
learn = Learner(dls, model, metrics=accuracy)
learn.fit_one_cycle(3, 1e-4)

# 4. Quantize for deployment
from fasterai.quantize.quantizer import Quantizer
model = Quantizer(backend='x86', method='static').quantize(model, dls.valid)
```

### Recommended ordering:
```
BN Fold → Tucker Decompose → Fine-tune → Prune → Quantize
```

---

## Summary

| Method | Layers | What it decomposes | Best for |
|--------|--------|-------------------|----------|
| `'tucker'` | 3 | Both channel dims | General purpose (default) |
| `'svd'` | 2 | Output channels | Moderate compression, less overhead |
| `'spatial'` | 2 | Kernel K×K → K×1 + 1×K | Small kernels (3×3, 5×5) |
| `'cp'` | 4 | Channels + spatial | Maximum compression |

| Feature | Description |
|---------|-------------|
| `Conv_Decomposer().decompose(model, 0.5)` | Tucker decomposition (default) |
| `method='svd'\|'tucker'\|'spatial'\|'cp'` | Choose decomposition method |
| `energy_threshold=0.99` | Auto rank selection (keep 99% energy) |
| `layers=['layer1'], exclude=['conv1']` | Per-layer control |

---

## See Also

- [FC Decomposer](tutorial.fc_decomposer.html) - SVD decomposition for Linear layers
- [BN Folding](bn_folding.html) - Fold BatchNorm before decomposition
- [Pruner Tutorial](../prune/pruner.html) - Apply after decomposition for further compression